In [37]:
import pandas as pd
import requests
import numpy as np
import re

# Acesso à Base de Dados
Neste notebook serão codificadas duas formas de acesso de dados, uma via requisição HTTP (busca no servidor) e a outra via arquivo direto guardado localmente na memória.

In [30]:
# Acessando a base de dados via requisição HTTP:
headers = {'User-Agent': 'Chrome/120.0.0.0'}
response = requests.get('https://cdn3.gnarususercontent.com.br/2928-transformacao-manipulacao-dados/dados_vendas_clientes.json', headers=headers)
data_frame = pd.read_json(response.text)

FileNotFoundError: File {"dados_vendas": [{"Data de venda": "06/06/2022", "Cliente": ["@ANA _LUCIA 321", "DieGO ARMANDIU 210", "DieGO ARMANDIU 210", "DieGO ARMANDIU 210"], "Valor da compra": ["R$ 836,5", "R$ 573,33", "R$ 392,8", "R$ 512,34"]}, {"Data de venda": "07/06/2022", "Cliente": ["Isabely JOanes 738", "Isabely JOanes 738", "Isabely JOanes 738", "Isabely JOanes 738"], "Valor da compra": ["R$ 825,31", "R$ 168,07", "R$ 339,18", "R$ 314,69"]}, {"Data de venda": "08/06/2022", "Cliente": ["Isabely JOanes 738", "JO\u00e3O Gabriel 671", "Julya meireles 914", "Julya meireles 914"], "Valor da compra": ["R$ 682,05", "R$ 386,34", "R$ 622,65", "R$ 630,79"]}, {"Data de venda": "09/06/2022", "Cliente": ["Julya meireles 914", "MaRIA Julia 444", "MaRIA Julia 444", "MaRIA Julia 444"], "Valor da compra": ["R$ 390,3", "R$ 759,16", "R$ 334,47", "R$ 678,78"]}, {"Data de venda": "10/06/2022", "Cliente": ["MaRIA Julia 444", "PEDRO PASCO 812", "Paulo castro 481", "Thiago fritzz 883"], "Valor da compra":

In [31]:
# Acessando via arquivo da base de dados armazenado localmente:
data_frame = pd.read_json('/home/felipe-rrgama/Área de trabalho/customer_highest_purchase_analisys/dataframe_archive/dados_vendas_clientes.json')

In [32]:
# Visualizando o dataframe:
data_frame.head(10)

,dados_vendas
0,"{'Data de venda': '06/06/2022', 'Cliente': ['@..."
1,"{'Data de venda': '07/06/2022', 'Cliente': ['I..."
2,"{'Data de venda': '08/06/2022', 'Cliente': ['I..."
3,"{'Data de venda': '09/06/2022', 'Cliente': ['J..."
4,"{'Data de venda': '10/06/2022', 'Cliente': ['M..."


In [33]:
# Como os dados estão aninhados, vamos normalizá-los:
data_frame = pd.json_normalize(data_frame['dados_vendas'])

In [34]:
data_frame = data_frame.explode(list(data_frame.columns[1:3]), ignore_index=True)

data_frame.head()

,Data de venda,Cliente,Valor da compra
0,06/06/2022,@ANA _LUCIA 321,"R$ 836,5"
1,06/06/2022,DieGO ARMANDIU 210,"R$ 573,33"
2,06/06/2022,DieGO ARMANDIU 210,"R$ 392,8"
3,06/06/2022,DieGO ARMANDIU 210,"R$ 512,34"
4,07/06/2022,Isabely JOanes 738,"R$ 825,31"


In [35]:
# Verificando os tipos de dados de cada coluna da tabela:
data_frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Data de venda    20 non-null     str  
 1   Cliente          20 non-null     str  
 2   Valor da compra  20 non-null     str  
dtypes: str(3)
memory usage: 612.0 bytes


In [36]:
# A coluna Valor da compra deve ser uma coluna numérica, logo, façamos o processo de transformação do tipo str para o tipo float32
data_frame['Valor da compra'] = data_frame['Valor da compra'].apply(lambda x: x.replace('R$ ', '').replace(',', '.').strip())

data_frame['Valor da compra'] = data_frame['Valor da compra'].astype(np.float32)

data_frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Data de venda    20 non-null     str    
 1   Cliente          20 non-null     str    
 2   Valor da compra  20 non-null     float32
dtypes: float32(1), str(2)
memory usage: 532.0 bytes


In [39]:
# Os valores da coluna 'Cliente' estão em um formato incomun para nomes de pessoas. Portanto, realizemos o processo de tratamento desses nomes, deixando-os em minúsculas e depois removendo os caracteres incomuns:
data_frame['Cliente'] = data_frame['Cliente'].str.lower()
data_frame['Cliente'] = data_frame['Cliente'].str.replace('[^a-zA-ZáàãâéêíóôõúüçÁÀÃÂÉÊÍÓÔÕÚÜÇ\s]', ' ', regex=True)

data_frame['Cliente']

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_5618/3273897261.py:3: SyntaxWarning: invalid escape sequence '\s'
  data_frame['Cliente'] = data_frame['Cliente'].str.replace('[^a-zA-ZáàãâéêíóôõúüçÁÀÃÂÉÊÍÓÔÕÚÜÇ\s]', ' ', regex=True)


0         ana  lucia    
1     diego armandiu    
2     diego armandiu    
3     diego armandiu    
4     isabely joanes    
5     isabely joanes    
6     isabely joanes    
7     isabely joanes    
8     isabely joanes    
9       joão gabriel    
10    julya meireles    
11    julya meireles    
12    julya meireles    
13       maria julia    
14       maria julia    
15       maria julia    
16       maria julia    
17       pedro pasco    
18      paulo castro    
19     thiago fritzz    
Name: Cliente, dtype: str